# DQL String Functions: REVERSE, REPLACE, and LENGTH

These functions reshape text and help check text quality.


## How to Think About String Functions

String functions clean, standardize, search, and reshape text. They are common in real data pipelines because text values often arrive with inconsistent casing, extra spaces, embedded codes, or formatting differences.

Common uses include:

* Standardizing names and categories.
* Removing leading or trailing spaces.
* Extracting prefixes, suffixes, and codes.
* Building readable labels from multiple columns.
* Finding or replacing text inside a value.

String cleanup is often done before joins, filters, and grouping so values match consistently.

## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## REVERSE, REPLACE, and LENGTH

These functions are useful for formatting and data quality checks.

These functions are often used for data cleanup and checking text quality.

`replace` changes one piece of text to another. `length` counts characters, which helps find unusually short or long values.


In [ ]:
emp.select(
    "ename",
    F.reverse("ename").alias("reversed_name"),
    F.regexp_replace("ename", "_", "-").alias("dash_name"),
    F.length("ename").alias("name_length")
).show(10)

spark.sql("""
SELECT ename,
       REVERSE(ename) AS reversed_name,
       REPLACE(ename, '_', '-') AS dash_name,
       LENGTH(ename) AS name_length
FROM emp
LIMIT 10
""").show()
